# FastGen Final Workflow

This notebook creates the FastGen environment, downloads CIFAR-10 data and FID reference stats, configures W&B, trains a student model, and evaluates both the teacher and student models.

In [ ]:
!conda create -y -n fastgen python=3.12.3 pip

In [ ]:
%%bash
source ~/.bashrc
conda activate fastgen
python --version

## Download CIFAR-10 Dataset

In [ ]:
!python scripts/download_data.py --dataset cifar10

## Download CIFAR-10 FID Reference Statistics

In [ ]:
%%bash
source ~/.bashrc
conda activate fastgen
!python scripts/download_data.py --dataset cifar10 --compute-fid-refs

## Configure W&B Token

In [ ]:
import os
token = '<YOUR_WANDB_API_KEY>'  # Replace with your actual W&B API key
os.makedirs('credentials', exist_ok=True)
with open('credentials/wandb_api.txt', 'w', encoding='utf-8') as f:
    f.write(token)
os.environ['WANDB_API_KEY'] = token
print('W&B token saved to credentials/wandb_api.txt')

## Train the Student Model

In [ ]:
import os
CONFIG = 'fastgen/configs/experiments/EDM/config_dmd2_cifar10.py'
LOG_NAME = 'student_run'
NUM_GPUS = 4
os.environ['FASTGEN_OUTPUT_ROOT'] = os.getenv('FASTGEN_OUTPUT_ROOT', 'FASTGEN_OUTPUT')
print(f'Training student model with log_config.name={LOG_NAME}')
!torchrun --nproc_per_node={NUM_GPUS} train.py --config={CONFIG} - trainer.ddp=True log_config.name={LOG_NAME}

## Evaluate Teacher Model (Original EDM)

In [ ]:
import os
import shutil
FASTGEN_OUTPUT_ROOT = os.getenv('FASTGEN_OUTPUT_ROOT', 'FASTGEN_OUTPUT')
src_path = os.path.join(FASTGEN_OUTPUT_ROOT, 'MODEL', 'cifar10', 'edm-cifar10-32x32-cond-vp.pth')
dst_dir = os.path.join(FASTGEN_OUTPUT_ROOT, 'fastgen', 'cifar10', 'EDM_original', 'checkpoints')
dst_path = os.path.join(dst_dir, '0000001.pth')
os.makedirs(dst_dir, exist_ok=True)
assert os.path.exists(src_path), f'Checkpoint not found: {src_path}'
shutil.copy2(src_path, dst_path)
print(f'Copied teacher checkpoint to: {dst_path}')
!python scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_sft_edm_cifar10.py log_config.name=EDM_original

## Evaluate Student Model

In [ ]:
import os
LOG_NAME = 'student_run'
print(f'Evaluating student model: {LOG_NAME}')
!python scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py log_config.name={LOG_NAME}